In [ ]:
import warnings
warnings.filterwarnings("ignore", message="pkg_resources is deprecated as an API.*", category=UserWarning)

from adamacs.notebook_runtime import bootstrap_ingest_notebook

ctx = bootstrap_ingest_notebook(verbose=True)
repo_root = ctx.repo_root

import datajoint as dj


In [ ]:
import os
# change to the upper level folder to detect dj_local_conf.json
from pathlib import Path
if Path.cwd().name == 'notebooks':
    os.chdir('..')
from adamacs.pipeline import subject, session, equipment, surgery, event, trial, imaging, behavior, scan, model, denoising
from adamacs.ingest import session as isess
from adamacs.ingest.harp import CamLoader_sync
from adamacs.helpers import stack_helpers as sh
from adamacs.helpers import trace_helpers as th
from adamacs.helpers import dj_helpers as djh
from adamacs.ingest import behavior as ibe
from adamacs.paths import get_experiment_root_data_dir
import datajoint as dj
from rspace_client.eln import eln
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from datetime import datetime
from scipy.spatial.transform import Rotation as R
from element_interface.utils import find_full_path
print(dj.__version__)
print(dj.config['custom']['database.prefix'])

In [ ]:
from adamacs.schemas import mocap

In [ ]:
# mocap.schema.drop()

In [ ]:
import warnings

warnings.filterwarnings("ignore")
dj.Diagram(mocap)

# Test Ingest

In [ ]:
# first define a key to be used across multiple tables

scansi = "scan9FTZL0PY"
# scansi = "scan9FU4ATBA"
# scansi = "scan9FU63S5H"

scan_key = (scan.Scan & f'scan_id = "{scansi}"').fetch('KEY')[0]
# curation_key = (imaging.Curation & scan_key & 'curation_id=1').fetch1('KEY')
sessi = (scan.Scan & f'scan_id = "{scansi}"').fetch('session_id')[0]
aux_setup_typestr = (scan.ScanInfo() & scan_key).fetch("userfunction_info")[0] # check setup type (not needed)
print(aux_setup_typestr)
print((scan.ScanPath & scan_key).fetch("path")[0])

In [ ]:
key = (scan.Scan * session.SessionUser * subject.User & "initials LIKE '%NK'").fetch('KEY')
scan_keys = (session.Session * scan.Scan & key & 'session_datetime > "2025-03-01"').fetch('KEY')
# scan_keys = (session.Session * scan.Scan & scan_key & 'session_datetime > "2025-03-01"').fetch('KEY')
# key = (imaging.Curation & key & 'curation_id >= 1').fetch('KEY')
# prockey = (imaging.Curation & key & 'paramset_idx = 7').fetch('KEY')
# prockey = (scan.Scan & key & 'scanner = "bench2p"').fetch('KEY')

In [ ]:
# DISK healed data would be inserted here as well
mocap.Mocap.insert1({
    "mocap_name": "motive_raw_data",
    "description": "The unprocessed export from Motive, with markers and rigid bodies",
}, skip_duplicates=True)
mocap.Mocap()


In [ ]:
(mocap.MocapRecording & scan_key).delete()

In [ ]:
import pathlib

selected_mocap_model = 'motive_raw_data'
search_str = 'ROS'
camera = 'mocap'
populate_settings = {'display_progress': True, 'suppress_errors': False, 'processes': 1}

for keylo in scan_keys:
    try:
        # Get the movie file path
        mocappath = str(list(pathlib.Path((scan.ScanPath() & keylo).fetch("path")[0]).glob(f"*{search_str}*.tak*"))[0])
        # Update key with camera info
        keylo.update({'camera': camera})
        
        # Insert into MocapRecording table
        mocap.MocapRecording.insert1(keylo, skip_duplicates=True, ignore_extra_fields=True)
        
        # Update key with file info
        keylo.update({
            'file_path': mocappath,
            'file_id': 0  # Adjust FILE_ID with camera number if needed
        })
        
        # Insert into MocapRecording.File table and populate recording info
        mocap.MocapRecording.File.insert1(keylo, ignore_extra_fields=True, skip_duplicates=True)
        mocap.MocapRecordingInfo.populate(keylo, **populate_settings)
        keylo.update({'mocap_name': selected_mocap_model}) 
        mocap.MotionCaptureTask.insert1(keylo, ignore_extra_fields=True, skip_duplicates=True)
    except Exception as e:
        print(f"Error processing {keylo}: {e}")
        continue

In [ ]:
populate_settings = {'display_progress': True, 'suppress_errors': True, 'processes': 1}
errs = mocap.MotionCapture.populate(scan_keys,**populate_settings, return_exception_objects=True)
errs

In [ ]:
 (mocap.MocapRecordingInfo & scan_keys).fetch1("metadata")

In [ ]:
mocap.MocapRecording.File.insert1(keylo, ignore_extra_fields=True, skip_duplicates=True)

In [ ]:
mocap.MocapRecording.File & scan_key

In [ ]:
mocap.MocapRecordingInfo & keylo

In [ ]:
mocap.MocapRecording().delete()

In [ ]:
mocap.MocapRecording.File()

In [ ]:
mocap.MocapRecordingInfo()

In [ ]:
dirc = mocap.MotionCapture.MocapRecordingInfo().fetch1("metadata")
dirc

In [ ]:
key = scan_key.copy()
key.update({'mocap_name': selected_mocap_model}) 

In [ ]:
key

In [ ]:
# INSERT mocap task

dj.logger.setLevel('INFO')

In [ ]:
mocap.MotionCaptureTask().delete()

In [ ]:
errs

In [ ]:
(mocap.MotionCapture & scan_key).delete()

In [ ]:
mocap.Mocap.TrackingId() & key

In [ ]:
mocap.MotionCapture.TrackingPosition() & scan_key

In [ ]:
(mocap.MotionCapture & scan_key).delete()

In [ ]:
mocap.MotionCapture.RigidBodyPosition() & scan_key
# key = scan_key

In [ ]:
rec_file = (mocap.MocapRecording.File & scan_key).fetch1("file_path")
rec_stem = Path(rec_file).with_suffix("")
override = (mocap.MotionCaptureTask & scan_key).fetch1("csv_path")
csv_path = Path(override) if override else rec_stem.with_suffix(".csv")

frame_idx, ts, markers, marker_uids, rigid_bodies, rb_uids = mocap._parse_motive_csv(csv_path)

In [ ]:
(mocap.MotionCapture.RigidBodyPosition() & key & "rigid_body = 'Mouse_trackers'")

In [ ]:
key = scan_keys[0]

import plotly.graph_objects as go

# Extract the inner arrays
xpos = (mocap.MotionCapture.RigidBodyPosition() & key & "rigid_body = 'Mouse_trackers'").fetch1('x_pos')
zpos = (mocap.MotionCapture.RigidBodyPosition() & key & "rigid_body = 'Mouse_trackers'").fetch1('y_pos')
ypos = (mocap.MotionCapture.RigidBodyPosition() & key & "rigid_body = 'Mouse_trackers'").fetch1('z_pos')
time_arr = (mocap.MotionCapture.RigidBodyPosition() & key).fetch('timestamps')[1]

# Compute speed (Euclidean distance over dt)
dx = np.diff(xpos)
dy = np.diff(ypos)
dz = np.diff(zpos)
dt_local = np.diff(time_arr)
speed = np.sqrt(dx**2 + dy**2 + dz**2) / dt_local
speed = np.concatenate(([0], speed))

# Set color limits
vmin = 0
vmax = 500

fig = go.Figure()

fig.add_trace(go.Scatter3d(
    x=xpos, y=ypos, z=zpos,
    mode='markers',
    marker=dict(
        size=2,
        color=speed,
        colorscale='Viridis',
        cmin=vmin,
        cmax=vmax,
        opacity=0.1,
        colorbar=dict(title='Speed')
    )
))

fig.update_layout(
    title="3D Position Colored by Speed",
    width=1200, height=1200,
    scene=dict(
        xaxis_title='X Position',
        yaxis_title='Y Position',
        zaxis_title='Z Position'
    )
)

fig.show()


In [ ]:
# Retrieve the roll, pitch, and yaw for the rigid body from the current scan
roll = (mocap.MotionCapture.RigidBodyPosition & scan_key).fetch1("roll")
pitch = (mocap.MotionCapture.RigidBodyPosition & scan_key).fetch1("pitch")
yaw = (mocap.MotionCapture.RigidBodyPosition & scan_key).fetch1("yaw")
time = (mocap.MotionCapture.RigidBodyPosition & scan_key).fetch1("timestamps")
z_pos = (mocap.MotionCapture.RigidBodyPosition & scan_key).fetch1("z_pos")

# Convert angles to degrees
roll_deg = np.degrees(roll)
pitch_deg = np.degrees(pitch)
yaw_deg = np.degrees(yaw)

# Interactive Plotly version for better visualization
import plotly.graph_objects as go

fig = go.Figure()
fig.add_trace(go.Scatter(x=time, y=roll_deg, mode='lines', name='Roll (x)'))
fig.add_trace(go.Scatter(x=time, y=pitch_deg, mode='lines', name='Pitch (y)'))
fig.add_trace(go.Scatter(x=time, y=yaw_deg, mode='lines', name='Yaw (z)'))
fig.add_trace(go.Scatter(x=time, y=z_pos, mode='lines', name='Z Position'))

fig.update_layout(
    title='Euler Angles Over Time (Rigid Body from Database)',
    xaxis_title='Time (s)',
    yaxis_title='Angle (degrees)',
    width=1200,
    height=600
)

fig.show()


In [ ]:
from scipy.signal import welch, spectrogram
import pandas as pd

# Convert pitch to a pandas Series and interpolate missing values (linear interpolation)
pitch_series = pd.Series(pitch)
if pitch_series.isna().all():
    raise ValueError("The pitch signal is entirely NaN.")
pitch_interp = pitch_series.interpolate(method='linear', limit_direction='both').to_numpy()

# Similarly, if dt contains NaNs, interpolate to estimate the sampling intervals
dt_series = pd.Series(dt)
if dt_series.isna().all():
    raise ValueError("The dt array is entirely NaN.")
dt_interp = dt_series.interpolate(method='linear', limit_direction='both').to_numpy()

# Calculate the sampling frequency from the interpolated dt values
fs = 1 / np.nanmean(dt_interp)

# Compute power spectral density using Welch’s method on the interpolated pitch signal
f_psd, Pxx = welch(pitch_interp, fs=fs, nperseg=1024)

plt.figure(figsize=(12, 5))

# Plot the power spectral density
plt.subplot(1, 2, 1)
plt.semilogy(f_psd, Pxx)
plt.xlabel('Frequency (Hz)')
plt.ylabel('Power Spectral Density')
plt.title('PSD of Pitch Signal')

# Compute the spectrogram of the interpolated pitch signal
f_spec, t_spec, Sxx = spectrogram(pitch_interp, fs=fs, nperseg=256)

# Plot the spectrogram (in dB)
plt.subplot(1, 2, 2)
plt.pcolormesh(t_spec, f_spec, 10 * np.log10(Sxx), shading='gouraud')
plt.xlabel('Time (s)')
plt.ylabel('Frequency (Hz)')
plt.title('Spectrogram of Pitch Signal')
plt.colorbar(label='Power/Frequency (dB/Hz)')

plt.tight_layout()
plt.show()

In [ ]:
# Debug: auto-detect Motive header rows and inspect parsed columns
from pathlib import Path
import pandas as pd

csv_path = Path("/datajoint-data/data/nataliak/NK_ROS-2076_2025-05-05_scan9FTZL0PY_sess9FTZL0PY/scan9FTZL0PY_ROS-2076_2025-05-05_002.csv")

with open(csv_path, 'r', encoding='utf-8', errors='ignore') as fh:
    peek = [fh.readline().rstrip('\n') for _ in range(25)]

frame_line_idx = next((i for i, ln in enumerate(peek) if ln.strip().startswith('Frame,')), None)
if frame_line_idx is None:
    raise ValueError('Frame header not found')

header_start_idx = next((i for i, ln in enumerate(peek[:frame_line_idx])
                        if any(tok.strip() == 'Type' for tok in ln.split(',') if tok.strip())), None)
if header_start_idx is None:
    raise ValueError('Type header not found')

header_rows = list(range(frame_line_idx - header_start_idx + 1))
print(f'Detected header rows: {header_rows}, skiprows={header_start_idx}')

df = pd.read_csv(csv_path, skiprows=header_start_idx, header=header_rows, low_memory=False, nrows=20)
print('Shape:', df.shape)

col_map = {tuple(col[-5:]): col for col in df.columns}
frame_col = next(col for col in col_map if any('Frame' == str(l) for l in col))
time_col = next(col for col in col_map if any('Time (Seconds)' == str(l) for l in col))
print('Frame col:', frame_col)
print('Time col:', time_col)
# display(df[[col_map[frame_col], col_map[time_col]]].head())
display(df.head())

In [ ]:
# Debug: legacy fixed header read for comparison on the same CSV
import pandas as pd
from pathlib import Path

csv_path = Path("/datajoint-data/data/nataliak/NK_ROS-2076_2025-05-05_scan9FTZL0PY_sess9FTZL0PY/scan9FTZL0PY_ROS-2076_2025-05-05_002.csv")
df_old = pd.read_csv(csv_path, skiprows=2, header=[0,1,2,3,4,5], low_memory=False, nrows=20)
df_old.columns = df_old.columns.droplevel(-3)
print('Old parser shape:', df_old.shape)
display(df_old.head())
for col in df_old.columns:
    ctype, name, _id, meas, axis = col
ctype, name, _id, meas, axis

In [ ]:
df_old[0].to_numpy(float, copy=True) 

In [ ]:
import pandas as pd
from pathlib import Path

# csv_path = Path("/datajoint-data/data/nataliak/NK_ROS-2076_2025-05-05_scan9FTZL0PY_sess9FTZL0PY/scan9FTZL0PY_ROS-2076_2025-05-05_002.csv")
csv_path = Path("/datajoint-data/data/nataliak/NK_ROS-2075_2025-05-13_scan9FU4ATBA_sess9FU4ATBA/scan9FU4ATBA_ROS-2075_2025-05-13_003.csv")
    
df_old = pd.read_csv(csv_path, skiprows=2, header=[0,1,2,3,4], low_memory=False, nrows=20)
print('Old parser shape:', df_old.shape)
display(df_old.head())
for col in df_old.columns:
    ctype, name, _id, meas, axis = col
ctype, name, _id, meas, axis

In [ ]:
ctype, name, _id, meas, axis

In [ ]:
# Plot the Rigid Body Position in 3D interactively with Plotly (larger figure)
import plotly.graph_objects as go

# markerselect = 'Unlabeled 3698'  # Change this to the desired marker name
# Find all columns that ma

# Find all Rigid Body Position columns (assuming levels contain 'Rigid Body' and 'Position')
position_cols = [col for col in df.columns if (markerselect in str(col) and 'Position' in str(col))]

if position_cols:
    from collections import defaultdict
    rb_groups = defaultdict(list)
    for col in position_cols:
        name = next((level for level in col if markerselect in str(level)), str(col))
        rb_groups[name].append(col)
    first_rb = next(iter(rb_groups.values()))
    xyz_cols = sorted(first_rb, key=lambda c: ['X','Y','Z'].index([l for l in c if l in ['X','Y','Z']][0]) if any(l in ['X','Y','Z'] for l in c) else 0)
    xyz = df[xyz_cols].values
    fig = go.Figure(data=[go.Scatter3d(x=xyz[:,0], y=xyz[:,1], z=xyz[:,2],
                                       mode='lines',
                                       line=dict(color='blue', width=2),
                                       opacity=0.5)])
    fig.update_layout(scene = dict(xaxis_title='X', yaxis_title='Y', zaxis_title='Z'),
                      title='Rigid Body Position in 3D (interactive)',
                      width=1200, height=900)
    fig.show()
else:
    print('No Rigid Body Position columns found.')

In [ ]:
from scipy.spatial.transform import Rotation as R
from collections import defaultdict

# Get Rigid Body quaternion columns
quat_cols = [col for col in df.columns if markerselect in str(col) and 'Rotation' in str(col)]
rb_quat_groups = defaultdict(list)
for col in quat_cols:
    key = next((level for level in col if markerselect in str(level)), str(col))
    rb_quat_groups[key].append(col)

if rb_quat_groups:
    first_rb = next(iter(rb_quat_groups.values()))
    order = ['X', 'Y', 'Z', 'W']
    # Order columns using the first level found in the "order" list
    quat_ordered = sorted(first_rb, key=lambda c: order.index(next(l for l in c if l in order)))
    quats = df[quat_ordered].values

    norms = np.linalg.norm(quats, axis=1)
    valid = norms > 1e-8
    euler = np.full((len(quats), 3), np.nan)
    if valid.any():
        euler[valid] = R.from_quat(quats[valid]).as_euler('xyz', degrees=True)
    
    euler_df = pd.DataFrame(euler, columns=['roll (x)', 'pitch (y)', 'yaw (z)'])
    display(euler_df.head())
    
    if not valid.all():
        print(f"Info: {np.sum(~valid)} rows with zero-norm quaternions are set to NaN in Euler output.")
else:
    print('No Rigid Body quaternion columns found.')


In [ ]:
# Plot Euler angles (roll, pitch, yaw) over time for the first rigid body interactively with Plotly

time_col = [col for col in df.columns if 'Time' in str(col)]
if rb_quat_groups and time_col:
    time = df[time_col[0]].values
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=time, y=euler_df['roll (x)'],
                             mode='lines',
                             name='Roll (x)'))
    fig.add_trace(go.Scatter(x=time, y=euler_df['pitch (y)'],
                             mode='lines',
                             name='Pitch (y)'))
    fig.add_trace(go.Scatter(x=time, y=euler_df['yaw (z)'],
                             mode='lines',
                             name='Yaw (z)'))
    fig.update_layout(
        title='Euler Angles Over Time (First Rigid Body)',
        xaxis_title='Time (s)',
        yaxis_title='Angle (degrees)',
        width=1200,
        height=600
    )
    fig.show()
else:
    print('No Euler angles or time column found.')

In [ ]:
import plotly.graph_objects as go
from ipywidgets import interact, IntSlider
import numpy as np
from collections import defaultdict

markerselect = 'Rigid Body'  # or set to your marker name
rigid_marker_cols = [col for col in df.columns if any(markerselect in str(level) for level in col)]
df_rigid_markers = df[rigid_marker_cols]
position_cols = [col for col in df_rigid_markers.columns if 'Position' in str(col)]
rb_groups = defaultdict(list)
for col in position_cols:
    name = next((level for level in col if markerselect in str(level)), str(col))
    rb_groups[name].append(col)

def plot_rigid_body_timepoint(frame=0, fade=240):
    fig = go.Figure()
    for rb_name, cols in rb_groups.items():
        xyz_cols = sorted(cols, key=lambda c: ['X','Y','Z'].index([l for l in c if l in ['X','Y','Z']][0]) if any(l in ['X','Y','Z'] for l in c) else 0)
        xyz = df[xyz_cols].values
        # Plot all marker positions at this frame as connected dots
        fig.add_trace(go.Scatter3d(x=xyz[frame,0:1], y=xyz[frame,1:2], z=xyz[frame,2:3],
                                   mode='markers+lines',
                                   marker=dict(size=6),
                                   name=f'{rb_name} marker'))
        # Plot the fading trace of the center
        center = xyz[:, :3]
        start = max(0, frame-fade)
        trace_xyz = center[start:frame+1]
        fig.add_trace(go.Scatter3d(x=trace_xyz[:,0], y=trace_xyz[:,1], z=trace_xyz[:,2],
                                   mode='lines',
                                   line=dict(color='blue', width=4),
                                   opacity=0.5,
                                   name=f'{rb_name} center trace'))
    fig.update_layout(scene = dict(xaxis_title='X', yaxis_title='Y', zaxis_title='Z'),
                      title=f'Rigid Body Markers at Frame {frame}',
                      width=900, height=700)
    fig.show()

n_frames = len(df)
interact(plot_rigid_body_timepoint, frame=IntSlider(min=0, max=n_frames-1, step=1, value=0), fade=IntSlider(min=1, max=500, step=1, value=240))

### Deep Labcut

In [ ]:
scan_key

In [ ]:
model.BodyPart() & scan_key

In [ ]:
 model.PoseEstimationNew.BodyPartPosition & scan_key

In [ ]:
model.RecordingInfoNew() & scan_key

In [ ]:
dlc_scan_key = (model.PoseEstimationNew & scan_key).fetch('KEY')
path = (model.VideoRecordingNew.File & scan_key).fetch("file_path")
path

In [ ]:
model.BodyParts

In [ ]:
#reduce dataframe to xy coordinates
# dlc_scan_key =dlc_scan_key[0]
df=model.PoseEstimationNew.get_trajectory(dlc_scan_key[0])
df_xy = df.iloc[:,df.columns.get_level_values(2).isin(["x","y"])][dlc_scan_key[0]['model_name']]
# df_xy.mean()
# df_xy
df_xy.plot().legend(loc='right')
plt.show()

In [ ]:
df_flat = df_xy.copy()
df_flat.columns = df_flat.columns.map('_'.join)

fig,ax=plt.subplots()
# df_flat.plot(x='body_middle_x',y='body_middle_y',ax=ax)
df_flat.plot(x='head_middle_x',y='head_middle_y', ax=ax)
# df_flat.plot(x='tail_x',y='tail_y', ax=ax)
ax.set_aspect('equal')
plt.title(scan_key)
plt.show()